In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import uuid

In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyecto")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv"

tabla_destino = f"{catalogo}.{esquema}.weather_hourly"

source_system = "KAGGLE_NYC_WEATHER"

batch_id = str(uuid.uuid4())

print(f"Ruta origen    : {ruta}")
print(f"Tabla destino  : {tabla_destino}")
print(f"Source system  : {source_system}")
print(f"Batch ID       : {batch_id}")

Ruta origen    : abfss://raw@adlsproyecto.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv
Tabla destino  : catalog_au.bronze.weather_hourly
Source system  : KAGGLE_NYC_WEATHER
Batch ID       : b1fa4772-9769-4564-8dcd-1ec6ab020a85


In [0]:
df_weather = spark.read.option('header', True)\
                       .option('inferSchema', True)\
                       .csv(ruta)

df_weather.printSchema()

print("Columnas RAW:")
for columna in df_weather.columns:
    print(columna)

root
 |-- time: timestamp (nullable = true)
 |-- temperature_2m (°C): double (nullable = true)
 |-- precipitation (mm): double (nullable = true)
 |-- rain (mm): double (nullable = true)
 |-- cloudcover (%): double (nullable = true)
 |-- cloudcover_low (%): double (nullable = true)
 |-- cloudcover_mid (%): double (nullable = true)
 |-- cloudcover_high (%): double (nullable = true)
 |-- windspeed_10m (km/h): double (nullable = true)
 |-- winddirection_10m (°): double (nullable = true)

Columnas RAW:
time
temperature_2m (°C)
precipitation (mm)
rain (mm)
cloudcover (%)
cloudcover_low (%)
cloudcover_mid (%)
cloudcover_high (%)
windspeed_10m (km/h)
winddirection_10m (°)


In [0]:
weather_schema = StructType(fields=[
    StructField("time", TimestampType(), True),
    StructField("temperature_2m (°C)", DoubleType(), True),
    StructField("precipitation (mm)", DoubleType(), True),
    StructField("rain (mm)", DoubleType(), True),
    StructField("cloudcover (%)", DoubleType(), True),
    StructField("cloudcover_low (%)", DoubleType(), True),
    StructField("cloudcover_mid (%)", DoubleType(), True),
    StructField("cloudcover_high (%)", DoubleType(), True),
    StructField("windspeed_10m (km/h)", DoubleType(), True),
    StructField("winddirection_10m (°)", DoubleType(), True)
])

In [0]:
df_weather_final = spark.read\
    .option('header', True)\
    .schema(weather_schema)\
    .csv(ruta)\
    .select(
        "*",
        col("_metadata.file_path").alias("_source_file")
    )

In [0]:
weather_selected_df = df_weather_final.select(
    col("time"),
    col("temperature_2m (°C)"),
    col("precipitation (mm)"),
    col("rain (mm)"),
    col("cloudcover (%)"),
    col("cloudcover_low (%)"),
    col("cloudcover_mid (%)"),
    col("cloudcover_high (%)"),
    col("windspeed_10m (km/h)"),
    col("winddirection_10m (°)"),
    col("_source_file")
)

In [0]:
weather_renamed_df = weather_selected_df\
    .withColumnRenamed("temperature_2m (°C)", "temperature_2m_c")\
    .withColumnRenamed("precipitation (mm)", "precipitation_mm")\
    .withColumnRenamed("rain (mm)", "rain_mm")\
    .withColumnRenamed("cloudcover (%)", "cloudcover_pct")\
    .withColumnRenamed("cloudcover_low (%)", "cloudcover_low_pct")\
    .withColumnRenamed("cloudcover_mid (%)", "cloudcover_mid_pct")\
    .withColumnRenamed("cloudcover_high (%)", "cloudcover_high_pct")\
    .withColumnRenamed("windspeed_10m (km/h)", "windspeed_10m_kmh")\
    .withColumnRenamed("winddirection_10m (°)", "winddirection_10m_deg")

In [0]:
weather_final_df = weather_renamed_df\
    .withColumn("_ingestion_timestamp", current_timestamp())\
    .withColumn("_source_system", lit(source_system))\
    .withColumn("_batch_id", lit(batch_id))

weather_final_df = weather_final_df.select(
    col("time"),
    col("temperature_2m_c"),
    col("precipitation_mm"),
    col("rain_mm"),
    col("cloudcover_pct"),
    col("cloudcover_low_pct"),
    col("cloudcover_mid_pct"),
    col("cloudcover_high_pct"),
    col("windspeed_10m_kmh"),
    col("winddirection_10m_deg"),
    col("_ingestion_timestamp"),
    col("_source_file"),
    col("_source_system"),
    col("_batch_id")
)

In [0]:
weather_final_df.select(
    count("*").alias("total_registros"),
    count("time").alias("time_validos"),
    count("temperature_2m_c").alias("temperature_validos"),
    count("precipitation_mm").alias("precipitation_validos"),
    count("rain_mm").alias("rain_validos"),
    count("cloudcover_pct").alias("cloudcover_validos"),
    count("windspeed_10m_kmh").alias("windspeed_validos"),
    count("winddirection_10m_deg").alias("winddirection_validos")
).show()

+---------------+------------+-------------------+---------------------+------------+------------------+-----------------+---------------------+
|total_registros|time_validos|temperature_validos|precipitation_validos|rain_validos|cloudcover_validos|windspeed_validos|winddirection_validos|
+---------------+------------+-------------------+---------------------+------------+------------------+-----------------+---------------------+
|          59760|       59760|              59760|                59760|       59760|             59760|            59760|                59760|
+---------------+------------+-------------------+---------------------+------------+------------------+-----------------+---------------------+



In [0]:
weather_final_df.select(
    min("time").alias("fecha_minima"),
    max("time").alias("fecha_maxima")
).show()

+-------------------+-------------------+
|       fecha_minima|       fecha_maxima|
+-------------------+-------------------+
|2016-01-01 00:00:00|2022-10-25 23:00:00|
+-------------------+-------------------+



In [0]:
weather_final_df.select(
    count("*").alias("total_registros"),

    sum(
        when(
            col("temperature_2m_c").isNull() |
            isnan(col("temperature_2m_c")),
            1
        ).otherwise(0)
    ).alias("temperature_faltantes"),

    sum(
        when(
            col("precipitation_mm").isNull() |
            isnan(col("precipitation_mm")),
            1
        ).otherwise(0)
    ).alias("precipitation_faltantes"),

    sum(
        when(
            col("rain_mm").isNull() |
            isnan(col("rain_mm")),
            1
        ).otherwise(0)
    ).alias("rain_faltantes"),

    sum(
        when(
            col("cloudcover_pct").isNull() |
            isnan(col("cloudcover_pct")),
            1
        ).otherwise(0)
    ).alias("cloudcover_faltantes"),

    sum(
        when(
            col("windspeed_10m_kmh").isNull() |
            isnan(col("windspeed_10m_kmh")),
            1
        ).otherwise(0)
    ).alias("windspeed_faltantes"),

    sum(
        when(
            col("winddirection_10m_deg").isNull() |
            isnan(col("winddirection_10m_deg")),
            1
        ).otherwise(0)
    ).alias("winddirection_faltantes")
).show()

+---------------+---------------------+-----------------------+--------------+--------------------+-------------------+-----------------------+
|total_registros|temperature_faltantes|precipitation_faltantes|rain_faltantes|cloudcover_faltantes|windspeed_faltantes|winddirection_faltantes|
+---------------+---------------------+-----------------------+--------------+--------------------+-------------------+-----------------------+
|          59760|                  168|                    168|           168|                 168|                168|                    173|
+---------------+---------------------+-----------------------+--------------+--------------------+-------------------+-----------------------+



In [0]:
weather_final_df.write.mode("overwrite")\
    .insertInto(tabla_destino)

In [0]:
total_raw = df_weather_final.count()
total_bronze = spark.table(tabla_destino).count()

print(f"Registros RAW    : {total_raw:,}")
print(f"Registros BRONZE : {total_bronze:,}")

if total_raw != total_bronze:
    raise Exception(
        "La cantidad de registros RAW y BRONZE no coincide."
    )

print("Ingesta weather_hourly finalizada correctamente.")

Registros RAW    : 59,760
Registros BRONZE : 59,760
Ingesta weather_hourly finalizada correctamente.


In [0]:
display(
    spark.table(tabla_destino).limit(10)
)

time,temperature_2m_c,precipitation_mm,rain_mm,cloudcover_pct,cloudcover_low_pct,cloudcover_mid_pct,cloudcover_high_pct,windspeed_10m_kmh,winddirection_10m_deg,_ingestion_timestamp,_source_file,_source_system,_batch_id
2016-01-01T00:00:00Z,7.6,0.0,0.0,69.0,53.0,0.0,72.0,10.0,296.0,2026-08-13T18:23:23.061647Z,abfss://raw@adlsproyecto.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv,KAGGLE_NYC_WEATHER,b1fa4772-9769-4564-8dcd-1ec6ab020a85
2016-01-01T01:00:00Z,7.5,0.0,0.0,20.0,4.0,0.0,56.0,9.8,287.0,2026-08-13T18:23:23.061647Z,abfss://raw@adlsproyecto.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv,KAGGLE_NYC_WEATHER,b1fa4772-9769-4564-8dcd-1ec6ab020a85
2016-01-01T02:00:00Z,7.1,0.0,0.0,32.0,3.0,0.0,99.0,9.7,285.0,2026-08-13T18:23:23.061647Z,abfss://raw@adlsproyecto.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv,KAGGLE_NYC_WEATHER,b1fa4772-9769-4564-8dcd-1ec6ab020a85
2016-01-01T03:00:00Z,6.6,0.0,0.0,35.0,5.0,0.0,100.0,9.2,281.0,2026-08-13T18:23:23.061647Z,abfss://raw@adlsproyecto.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv,KAGGLE_NYC_WEATHER,b1fa4772-9769-4564-8dcd-1ec6ab020a85
2016-01-01T04:00:00Z,6.3,0.0,0.0,34.0,4.0,0.0,100.0,9.1,279.0,2026-08-13T18:23:23.061647Z,abfss://raw@adlsproyecto.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv,KAGGLE_NYC_WEATHER,b1fa4772-9769-4564-8dcd-1ec6ab020a85
2016-01-01T05:00:00Z,6.1,0.0,0.0,35.0,5.0,0.0,100.0,9.4,277.0,2026-08-13T18:23:23.061647Z,abfss://raw@adlsproyecto.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv,KAGGLE_NYC_WEATHER,b1fa4772-9769-4564-8dcd-1ec6ab020a85
2016-01-01T06:00:00Z,6.0,0.0,0.0,50.0,21.0,1.0,100.0,9.7,274.0,2026-08-13T18:23:23.061647Z,abfss://raw@adlsproyecto.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv,KAGGLE_NYC_WEATHER,b1fa4772-9769-4564-8dcd-1ec6ab020a85
2016-01-01T07:00:00Z,5.9,0.0,0.0,51.0,24.0,0.0,98.0,9.7,272.0,2026-08-13T18:23:23.061647Z,abfss://raw@adlsproyecto.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv,KAGGLE_NYC_WEATHER,b1fa4772-9769-4564-8dcd-1ec6ab020a85
2016-01-01T08:00:00Z,5.8,0.0,0.0,54.0,26.0,1.0,99.0,9.0,265.0,2026-08-13T18:23:23.061647Z,abfss://raw@adlsproyecto.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv,KAGGLE_NYC_WEATHER,b1fa4772-9769-4564-8dcd-1ec6ab020a85
2016-01-01T09:00:00Z,5.8,0.0,0.0,58.0,31.0,1.0,99.0,10.2,262.0,2026-08-13T18:23:23.061647Z,abfss://raw@adlsproyecto.dfs.core.windows.net/weather/NYC_Weather_2016_2022.csv,KAGGLE_NYC_WEATHER,b1fa4772-9769-4564-8dcd-1ec6ab020a85


In [0]:
cantidad_archivos = (
    weather_final_df
    .select("_source_file")
    .distinct()
    .count()
)

print(f"Archivos procesados: {cantidad_archivos}")

if cantidad_archivos != 1:
    raise Exception(
        f"Se esperaba 1 archivo Weather y se encontraron {cantidad_archivos}."
    )

Archivos procesados: 1
